# Chapter 01. 베스트셀러 데이터 이해와 기본 전처리

교보문고 베스트셀러 Excel 데이터를 불러와 분석에 필요한 컬럼을 선택하고, 결측치·자료형·중복을 확인한 뒤 다음 텍스트 분석에 사용할 CSV 파일로 저장합니다.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

# VS Code/Jupyter의 실행 위치가 프로젝트 루트인지 Notebook 폴더인지 모두 대응
candidates = [
    Path("notebooks/book-text-ml/교보문고_종합_베스트셀러_상품리스트.xlsx"),
    Path("교보문고_종합_베스트셀러_상품리스트.xlsx"),
]

file_path = next((path for path in candidates if path.exists()), None)
if file_path is None:
    raise FileNotFoundError("교보문고 Excel 파일을 찾을 수 없습니다.")

df = pd.read_excel(file_path)

print(type(df))
print("불러온 파일:", file_path)
display(df.head())


## 실습 2. 데이터의 크기와 컬럼 확인하기


In [ ]:
print("shape:", df.shape)
print("\ncolumns:")
print(df.columns.tolist())
print("\ndtypes:")
print(df.dtypes)
print("\n결측치 개수:")
print(df.isna().sum())

display(df.head())


## 실습 3. 분석에 필요한 컬럼만 선택하기


In [ ]:
selected_columns = [
    "순위",
    "판매상품 ID",
    "상품명",
    "판매가",
    "인물",
    "출판사",
    "발행(출시)일자",
    "분야",
]

df_books = df[selected_columns].copy()

display(df_books.head())
print(df_books.columns.tolist())


## 실습 4. 컬럼 이름을 사용하기 쉽게 정리하기


In [ ]:
df_books = df_books.rename(
    columns={
        "판매상품 ID": "판매상품ID",
        "인물": "저자",
        "발행(출시)일자": "발행일",
    }
)

print(df_books.columns.tolist())


## 실습 5. 결측치 확인하고 필요한 값 채우기


In [ ]:
print("처리 전 결측치")
print(df_books.isna().sum())

author_missing_before = int(df_books["저자"].isna().sum())
category_missing_before = int(df_books["분야"].isna().sum())

df_books["저자"] = df_books["저자"].fillna("미상")
df_books["분야"] = df_books["분야"].fillna("미분류")

print("\n처리 후 결측치")
print(df_books.isna().sum())


## 실습 6. 판매가를 숫자로 바꾸기


In [ ]:
print("변환 전 dtype:", df_books["판매가"].dtype)

df_books["판매가"] = (
    df_books["판매가"]
    .astype("string")
    .str.replace(",", "", regex=False)
)
df_books["판매가"] = pd.to_numeric(df_books["판매가"], errors="coerce")

print("변환 후 dtype:", df_books["판매가"].dtype)
display(df_books["판매가"].head())
display(df_books["판매가"].describe())


## 실습 7. 발행일을 날짜형으로 바꾸기


In [ ]:
release_text = (
    df_books["발행일"]
    .astype("string")
    .str.replace(r"\\.0$", "", regex=True)
    .str.strip()
)

df_books["발행일"] = pd.to_datetime(
    release_text,
    format="%Y%m%d",
    errors="coerce",
)

display(df_books["발행일"].head())
print("dtype:", df_books["발행일"].dtype)


## 실습 8. 문자열 앞뒤 공백 정리하기


In [ ]:
string_columns = ["상품명", "저자", "출판사", "분야"]

for column in string_columns:
    df_books[column] = df_books[column].astype("string").str.strip()

for column in string_columns:
    print(f"[{column}]")
    print(df_books[column].head().tolist())


## 실습 9. 중복 데이터 확인하기


In [ ]:
row_duplicate_count = int(df_books.duplicated().sum())
product_id_duplicate_count = int(df_books["판매상품ID"].duplicated().sum())
title_duplicate_count = int(df_books["상품명"].duplicated().sum())

print("전체 행 완전 중복:", row_duplicate_count)
print("판매상품ID 중복:", product_id_duplicate_count)
print("상품명 중복:", title_duplicate_count)


## 실습 10. 전처리 결과 최종 검증하기


In [ ]:
print("shape:", df_books.shape)
print("\ncolumns:")
print(df_books.columns.tolist())
print("\ndtypes:")
print(df_books.dtypes)
print("\n결측치:")
print(df_books.isna().sum())
print("\n판매가 앞의 5개:")
print(df_books["판매가"].head().tolist())
print("\n발행일 앞의 5개:")
print(df_books["발행일"].head().tolist())
print("\n판매상품ID 중복 개수:", df_books["판매상품ID"].duplicated().sum())

display(df_books.head())


## 실습 11. 전처리 데이터 저장하기


In [ ]:
output_path = file_path.parent / "book_bestseller_clean.csv"
df_books.to_csv(output_path, index=False, encoding="utf-8-sig")

print("저장 완료:", output_path)
print("파일 존재 여부:", output_path.exists())


## 실습 12. 실제 실행 결과를 바탕으로 Markdown 요약 만들기


In [ ]:
summary_md = f"""
### 전처리 결과 요약

1. **무엇을 확인했는지**  
   원본 데이터의 크기, 컬럼, 자료형, 결측치와 중복 여부를 확인했습니다. 원본 데이터 크기는 **{df.shape[0]}행 × {df.shape[1]}열**이었고, 분석에는 **{df_books.shape[1]}개 컬럼**을 사용했습니다.

2. **어떤 전처리를 했는지**  
   저자 결측치 **{author_missing_before}개**는 `미상`, 분야 결측치 **{category_missing_before}개**는 `미분류`로 처리했습니다. 판매가는 숫자형으로, 발행일은 날짜형으로 변환했고 문자열 앞뒤 공백도 정리했습니다. 판매상품ID 중복은 **{product_id_duplicate_count}개**였습니다.

3. **다음 분석을 위한 상태**  
   필요한 컬럼과 자료형을 정리하고 최종 데이터를 `book_bestseller_clean.csv`로 저장했습니다. 다음 Chapter에서 상품명 단어 빈도와 텍스트 분석에 사용할 수 있는 상태가 되었습니다.
"""

display(Markdown(summary_md))


## 확인 문제

**질문 1. 왜 모든 원본 컬럼을 사용하지 않고 필요한 컬럼만 선택했나요?**  
분석에 필요하지 않은 컬럼까지 계속 가지고 있으면 데이터가 복잡해지기 때문에, 이번 분석에 필요한 정보만 남기기 위해서입니다.

**질문 2. 판매가가 문자열이라면 어떤 문제가 발생할 수 있나요?**  
문자열 상태에서는 평균이나 합계 같은 숫자 계산이 제대로 되지 않을 수 있습니다.

**질문 3. 결측치를 확인한 뒤 바로 모든 행을 삭제하면 안 되는 이유는 무엇인가요?**  
결측치가 있어도 다른 분석에는 사용할 수 있는 데이터일 수 있기 때문에 분석 목적을 먼저 확인해야 합니다.

**질문 4. 같은 상품명이 두 번 등장했다고 해서 바로 중복 데이터라고 판단하기 어려운 이유는 무엇인가요?**  
같은 제목이라도 다른 판본이나 개정판, 다른 상품일 수 있기 때문입니다.

**질문 5. 인공지능이 작성한 코드가 오류 없이 실행되었다면 그것만으로 분석이 끝난 것일까요?**  
아닙니다. 실제 출력값이 기대한 데이터인지 다시 확인해야 합니다.


## 제출 전 확인

- [ ] Excel 파일이 정상적으로 로드된다.
- [ ] 데이터 크기와 컬럼을 확인했다.
- [ ] 필요한 컬럼만 선택했다.
- [ ] 컬럼 이름을 정리했다.
- [ ] 결측치를 확인하고 처리했다.
- [ ] 판매가를 숫자형으로 변환했다.
- [ ] 발행일을 날짜형으로 변환했다.
- [ ] 문자열 앞뒤 공백을 정리했다.
- [ ] 중복 데이터를 확인했다.
- [ ] 최종 데이터를 다시 검증했다.
- [ ] `book_bestseller_clean.csv`를 저장했다.
- [ ] Notebook 전체를 처음부터 다시 실행했을 때 오류가 없다.
